# 01 — Explore ECOSoundSet
Inspect the dataset, understand class distribution, and identify which UK Orthoptera species have sufficient clips.

**Kernel:** `Python (orthoptera-training)`  
**Dataset:** `datasets/ecosoundset/` — download via `zenodo_get 15043892` if not present.

In [ ]:
import pandas as pd
import soundfile as sf
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent  # cwd relative to current notebook path
ECO_ROOT = PROJECT_ROOT / "datasets" / "ecosoundset"
INSECT_ROOT = PROJECT_ROOT / "datasets" / "insectset459"


## EcoSoundSet Exploration

In [ ]:
# ── Load annotations and filter to train split ────────────────────────────────
all_annot_eco = pd.read_csv(ECO_ROOT / "annotated_audio_segments.csv")
train = all_annot_eco[all_annot_eco["subset"] == "train"].copy()
print(f"Total annotations (train): {len(train)}")
print(f"Columns: {list(train.columns)}")
train.head()


In [ ]:
# ── All Orthoptera species in the train split ────────────────────────────────
orth_eco = train[train["label_category"] == "Orthoptera"]
print(f"Orthoptera species: {orth_eco['label'].nunique()}\n")
print(f"Annotations per species:\n\n{orth_eco['label'].value_counts().to_string()}")


In [ ]:
# ── Quantify Overlapping Annotations ──────────────────────────────────────────
labels_per_file = all_annot_eco.groupby("audio_segment_file_name")["label"].count()
print(f"{'Average annotations per file:':<30} {labels_per_file.mean()}")
print(f"{'Least annotations per file:':<30} {labels_per_file.min()}")
print(f"{'Least annotated file:':<30} {labels_per_file.idxmin()}")
print(f"{'Maximum annotations per file:':<30} {labels_per_file.max()}")
print(f"{'Most annotated file:':<30} {labels_per_file.idxmax()}")


In [ ]:
# ── Filter to UK target species ───────────────────────────────────────────────
# ECOSoundSet uses trinomial names; Meconema thalassinum is absent from both datasets.
UK_SPECIES = [
    "Chorthippus brunneus brunneus",           # Field Grasshopper
    "Pseudochorthippus parallelus parallelus", # Meadow Grasshopper
    "Omocestus viridulus",                     # Common Green Grasshopper
    "Tettigonia viridissima",                  # Great Green Bush-cricket
    "Roeseliana roeselii",                     # Roesel's Bush-cricket
    "Pholidoptera griseoaptera",               # Dark Bush-cricket
    "Leptophyes punctatissima",                # Speckled Bush-cricket
    "Gryllus campestris",                      # Field Cricket
]

orth_uk_eco = orth_eco[orth_eco["label"].isin(UK_SPECIES)].copy()
# Multiple annotations can appear per clips. Annotation count does not equal clip count.
print(f"ECOSoundSet UK species annotations: {len(orth_uk_eco)}\n")
print(f"Annotations per species:\n\n{orth_uk_eco['label'].value_counts().to_string()}")


In [ ]:
# ── Quantify overlapping Orthoptera annotations ───────────────────────────────
# Files with at least one UK target species annotation.
uk_files = set(orth_uk_eco["audio_segment_file_name"])

# Files with at least 2 distinct Orthoptera annotations.
multi_orth = all_annot_eco[all_annot_eco["label_category"] == "Orthoptera"].groupby("audio_segment_file_name")["label"].nunique()
multi_orth_files = multi_orth[multi_orth > 1].index

# Files in both.
mixed_files = uk_files.intersection(multi_orth_files)

print(f"{'Clips with different overlapping Orthoptera annotations:'} {len(mixed_files)}")

In [ ]:
# ── Visualise class balance ───────────────────────────────────────────────────
counts = orth_uk_eco["label"].value_counts()
short_names = [s.split()[-1] for s in counts.index]

fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(short_names, counts.values, color="#2d8a4e")
ax.set_xlabel("Annotations")
ax.set_title("UK Orthoptera — Annotations per Species (ECOSoundSet train split)")
ax.axvline(100, color="orange", linestyle="--", linewidth=1, label="100 annotation threshold")
ax.legend()
plt.tight_layout()
plt.show()

low = counts[counts < 50]
if len(low):
    print(f"\n⚠ Species with <50 annotations (consider InsectSet459 supplement):")
    print(low.to_string())


In [ ]:
# ── Inspect a clip — spectrogram preview ─────────────────────────────────────
# Set use_random to True for a randomised UK Orthoptera sample
# Set use_random to False to select any annotation from ECOSoundSet
use_random = False
selected_index = 12412  # CSV row number minus 1

if use_random:
    sample = orth_uk_eco.sample(1).iloc[0]
else:
    sample = all_annot_eco.loc[selected_index]

audio_path = next(ECO_ROOT.rglob(sample["audio_segment_file_name"]), None)

if audio_path and audio_path.exists():
    t_start = sample["annotation_initial_time"] - sample["audio_segment_initial_time"]
    t_end = sample["annotation_final_time"] - sample["annotation_initial_time"]
    y, sample_rate = librosa.load(audio_path, sr=None, offset=t_start, duration=t_end - t_start)
    print(f"{'Species:':<20}{sample['label']}")
    print(f"{'File:':<20}{sample['audio_segment_file_name']}")
    print(f"{'Annotation Start:':<20}{sample['annotation_initial_time']}")
    print(f"{'Annotation End:':<20}{sample['annotation_final_time']}")
    print(f"{'Sample Rate:':<20}{sample_rate} Hz")
    print(f"{'Duration:':<20}{sample['annotation_final_time'] - sample['annotation_initial_time']:.1f} s")

    fig, axes = plt.subplots(2, 1, figsize=(12, 6))
    librosa.display.waveshow(y, sr=sample_rate, ax=axes[0])
    axes[0].set_title(f"{sample['label']} — waveform")

    n_fft = min(2048, len(y))
    S = librosa.feature.melspectrogram(y=y, sr=sample_rate, n_mels=128, fmax=20000, n_fft=n_fft)
    S_db = librosa.power_to_db(S, ref=np.max)
    librosa.display.specshow(S_db, sr=sample_rate, x_axis="time", y_axis="mel",
                             fmax=20000, ax=axes[1], cmap="viridis")
    axes[1].set_title("Mel spectrogram")
    plt.colorbar(axes[1].collections[0], ax=axes[1], format="%+2.0f dB")
    plt.tight_layout()
    plt.show()
else:
    print(f"Audio file not found: {audio_path}")
